In [2]:
#importing necessary libraries + data
import pandas as pd


# Load raw data
disclosure_dtype_mapping = {
    'DisclosureId': 'string',
    'JobStartDate': 'string',
    'JobEndDate': 'string',
    'APINumber': 'string',
    'StateName': 'category',
    'CountyName': 'category',
    'OperatorName': 'category',
    'WellName': 'category',
    'Projection': 'category'
}
watersource_dtype_mapping = {
    'WaterSourceId': 'string',
    'DisclosureId': 'string',
    'APINumber': 'string',
    'StateName': 'category',
    'CountyName': 'category',
    'OperatorName': 'category',
    'WellName': 'category',
    'Description': 'category'
}
registry_dtype_mapping = {
    'DisclosureId': 'string',
    'JobStartDate': 'string',
    'JobEndDate': 'string',
    'APINumber': 'string',
    'StateName': 'category',
    'CountyName': 'category',
    'OperatorName': 'category',
    'WellName': 'category',
    'Projection': 'category',
    'PurposeId': 'string',
    'TradeName': 'category',
    'Supplier': 'category',
    'Purpose': 'category',
    'IngredientsId': 'string',
    'CASNumber': 'string',
    'IngredientName': 'category',
    'IngredientMSDS': 'bool',
    'ClaimantCompany': 'category'
}
# 244k data
disclosure_data_path = "/Users/alishbahfarooq/Desktop/Chirality Research/fracfocus_case_study/data/FracFocusCSV_Oil_Gas/DisclosureList_1.csv"
disclosure_df = pd.read_csv(disclosure_data_path, dtype = disclosure_dtype_mapping)

#20k data
watersource_data_path = "/Users/alishbahfarooq/Desktop/Chirality Research/fracfocus_case_study/data/FracFocusCSV_Oil_Gas/WaterSource_1.csv"
watersource_df = pd.read_csv(watersource_data_path, dtype = watersource_dtype_mapping)

#500k data each (~7.5 mill total)
registry_df_path = "/Users/alishbahfarooq/Desktop/Chirality Research/fracfocus_case_study/data/FracFocusCSV_Oil_Gas/FracFocusRegistry_1.csv"
registry_df_raw = pd.read_csv(registry_df_path)

/var/folders/yy/x8lf9r5s1fgcphj75fsn3vv80000gn/T/ipykernel_6118/1439818967.py:57: DtypeWarning: Columns (30) have mixed types. Specify dtype option on import or set low_memory=False.
  registry_df_raw = pd.read_csv(registry_df_path)


## Cleaning Disclosure Data

In [3]:
disclosure_df_clean = disclosure_df.copy()
def replace_3xxx_23xx(col):
    # Replacing 3xxx with 2xxx
    col= col.str.replace(
        r"\b3(\d{3})", 
        r"2\1", 
        regex=True 
    )
    # Replacing 23xx with 20xx
    col = col.str.replace(
        r"\b23(\d{2})", 
        r"20\1", 
        regex=True 
    )
    return col

disclosure_df_clean['JobStartDate'] = replace_3xxx_23xx(disclosure_df_clean['JobStartDate'])
disclosure_df_clean['JobEndDate'] = replace_3xxx_23xx(disclosure_df_clean['JobEndDate'])

In [4]:
disclosure_df_clean['JobStartDate'] = pd.to_datetime(disclosure_df_clean['JobStartDate'], format='%m/%d/%Y %I:%M:%S %p')
disclosure_df_clean['JobEndDate'] = pd.to_datetime(disclosure_df_clean['JobEndDate'], format='%m/%d/%Y %I:%M:%S %p')

In [ ]:
# Select rows where 'Longitude' is not between -180 and -163 (exclusive)
lower_limit = -180
upper_limit = -163

filtered_df = disclosure_df_clean.loc[(disclosure_df_clean['Longitude'] < lower_limit) | (disclosure_df_clean['Longitude'] > upper_limit)]

filtered_df

In [ ]:
# Select rows where 'Latitude' is not between 15 and 75 (exclusive)
lower_limit = 15
upper_limit = 75

filtered_df = disclosure_df_clean.loc[(disclosure_df_clean['Longitude'] >= lower_limit) & (disclosure_df_clean['Longitude'] <= upper_limit)]

filtered_df

In [5]:
# Removing rows where 'TotalBaseNonWaterVolume' is less than 0
disclosure_df_clean = disclosure_df_clean.loc[~(disclosure_df_clean['TotalBaseNonWaterVolume'] < 0)]

disclosure_df_clean

,DisclosureId,JobStartDate,JobEndDate,APINumber,StateName,CountyName,OperatorName,WellName,Latitude,Longitude,Projection,TVD,TotalBaseWaterVolume,TotalBaseNonWaterVolume,FFVersion,FederalWell,IndianWell
0,02cd05a7-4b73-4722-9fba-1f798842a879,1955-05-01 00:00:00,1955-05-01 00:00:00,42317372620000,Texas,Martin,Pioneer Natural Resources,Rogers 42 #5,32.283431,-101.906575,NAD27,NaN,NaN,NaN,1,False,False
1,416d8b17-822f-4743-8c79-baf015a6de24,1982-05-19 00:00:00,1982-05-19 00:00:00,49009219470000,Wyoming,Converse,"Chesapeake Operating, Inc.",WILLIAM VALENTINE 1,42.972810,-105.953840,NAD27,NaN,NaN,NaN,1,False,False
2,89f97571-945b-4473-8be8-2aca326f04ad,1995-02-07 00:00:00,1995-02-07 00:00:00,49009228850000,Wyoming,Converse,"Chesapeake Operating, Inc.",LIZARD HEAD 1-8H RE,42.851470,-105.411510,NAD27,NaN,NaN,NaN,1,False,False
3,876b362a-e057-4815-a45e-e311c5ae1c1f,1996-06-11 00:00:00,1996-06-11 00:00:00,42335355480000,Texas,Mitchell,Energen Resources Corporation,North Westbrook Unit/Well No. 3032,32.440260,-101.030130,NAD27,NaN,NaN,NaN,1,False,False
4,0902e8c7-593f-42e1-b02c-283b0edceea7,2001-12-13 00:00:00,2001-12-13 00:00:00,42395313840000,Texas,Robertson,XTO Energy/ExxonMobil,Olene Reagan 3-1,31.257980,-96.374820,NAD27,NaN,NaN,NaN,1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244090,483a0a4f-850b-41c8-b4ae-82b6cad038a5,2026-02-08 01:07:39,2026-02-25 09:08:40,30015541280000,New Mexico,Eddy,Chevron USA Inc.,SND Javelina Unit 605H,32.235491,-103.777374,WGS84,11676.92,19984922.0,0.0,4,True,False
244091,d61308ab-3e81-4201-96e8-bd6dd0e4cefa,2026-02-07 12:10:12,2026-02-25 09:08:40,30015541300000,New Mexico,Eddy,Chevron USA Inc.,SND Javelina Unit 606H,32.235490,-103.776727,WGS84,11767.99,20379245.0,0.0,4,True,False
244092,c80aa35d-a22b-4e11-8726-5106105453cb,2026-02-08 02:00:21,2026-02-26 12:07:51,30015543290000,New Mexico,Eddy,Chevron USA Inc.,SND Javelina Unit 504H,32.235490,-103.776808,WGS84,11508.88,18871246.0,0.0,4,True,False
244093,825a1cb2-9b67-4df3-ac18-34f10e45b482,2026-02-07 12:09:44,2026-02-26 12:07:51,30015543300000,New Mexico,Eddy,Chevron USA Inc.,SND Javelina Unit 604H,32.235491,-103.776969,WGS84,11786.24,19994318.0,0.0,4,True,False


In [6]:
# 15 NaN for JobStartDate
# 4 NaN for CountyName
# 34 NaN for Latitude, Longitude
# 30139 NaN for TVD
# 30168 NaN for TotalBaseWaterVolume
# 50325 NaN for TotalBaseNonWaterVolume
disclosure_df_clean['CountyName'] = disclosure_df_clean['CountyName'] = disclosure_df_clean['CountyName'].cat.add_categories('Unknown')
NaN_county_data = disclosure_df_clean[disclosure_df_clean['CountyName'].isna()]

disclosure_df_clean["CountyName"] = disclosure_df_clean["CountyName"].fillna("Unknown")
disclosure_df_clean.loc[disclosure_df_clean['CountyName'] == 'Unknown']

,DisclosureId,JobStartDate,JobEndDate,APINumber,StateName,CountyName,OperatorName,WellName,Latitude,Longitude,Projection,TVD,TotalBaseWaterVolume,TotalBaseNonWaterVolume,FFVersion,FederalWell,IndianWell
170896,19d84746-971b-47eb-ba07-4afac64f349a,2019-11-19 06:00:00,2019-11-19 06:00:00,03729439000000,Arkansas,Unknown,WFD Oil Corporation,Vanorsdale,0.123455,-0.123450,NAD27,2442.00,22134.0,0.0,3,False,False
180051,378dd9d9-e781-48c6-be7b-20c2b4e98ca4,2020-11-11 00:00:00,2020-12-01 00:00:00,43317428660000,Utah,Unknown,Endeavor Energy Resources,Rhea 1-6 Unit 1 #133,32.409144,-101.808518,NAD83,8262.00,17519292.0,0.0,3,False,False
197739,cc76eb03-bc67-40cb-ac53-06eca842ed2f,2022-05-28 23:17:00,2022-06-08 23:43:00,33610338000000,North Dakota,Unknown,Hunt Oil Company,TRULSON 156-90-11-14H-3,48.355506,-102.212124,NAD83,8872.40,13496802.0,0.0,3,False,False
197781,7e308cc2-dc57-4d7c-9964-369885613467,2022-05-28 21:18:00,2022-06-09 06:48:00,33610338100000,North Dakota,Unknown,Hunt Oil Company,PALERMO 156-90-2-31H-5,48.355506,-102.212330,NAD83,8782.65,14820967.0,0.0,3,False,False


In [14]:
# JobStartDate                  15
# Latitude                      34
# Longitude                     34
# TVD                        30139
# TotalBaseWaterVolume       30168 -> fill with average?
# TotalBaseNonWaterVolume    50325 -> see above
disclosure_df_clean.isna().sum()

DisclosureId                   0
JobStartDate                  15
JobEndDate                     0
APINumber                      0
StateName                      0
CountyName                     0
OperatorName                   0
WellName                       0
Latitude                      34
Longitude                     34
Projection                     0
TVD                        30139
TotalBaseWaterVolume       30168
TotalBaseNonWaterVolume    50325
FFVersion                      0
FederalWell                    0
IndianWell                     0
dtype: int64

## Cleaning WaterSource Data

In [7]:
watersource_df_clean = watersource_df.copy()
watersource_df_clean.dtypes

WaterSourceId    string[python]
DisclosureId     string[python]
APINumber        string[python]
StateName              category
CountyName             category
OperatorName           category
WellName               category
Description            category
Percent                 float64
dtype: object

In [8]:
watersource_columns_to_drop = ['APINumber',
                               'StateName',
                               'CountyName',
                               'OperatorName']
watersource_df_clean = watersource_df_clean.drop(columns=watersource_columns_to_drop)

In [ ]:
watersource_df_clean.isna().sum()

In [ ]:
watersource_df_clean["Percent"].describe()

## Registry Dataset Cleaning

In [9]:
registry_df = registry_df_raw.astype(registry_dtype_mapping)
registry_df.dtypes

DisclosureId               string[python]
JobStartDate               string[python]
JobEndDate                 string[python]
APINumber                  string[python]
StateName                        category
CountyName                       category
OperatorName                     category
WellName                         category
Latitude                          float64
Longitude                         float64
Projection                       category
TVD                               float64
TotalBaseWaterVolume              float64
TotalBaseNonWaterVolume           float64
FFVersion                           int64
FederalWell                          bool
IndianWell                           bool
PurposeId                  string[python]
TradeName                        category
Supplier                         category
Purpose                          category
IngredientsId              string[python]
CASNumber                  string[python]
IngredientName                   c

In [10]:
registry_df_clean = registry_df_raw.copy()

In [10]:
# NaN values in each column
# TVD                         30135
# TotalBaseWaterVolume        30157
# TotalBaseNonWaterVolume    134538
# TradeName                   85580
# Supplier                    84129
# Purpose                     16499
# IngredientsId               56265
# CASNumber                   65834
# IngredientName              56381
# IngredientCommonName       147775 -> remove
# PercentHighAdditive         57761
# PercentHFJob                58086
# IngredientComment          471882 -> remove
# IngredientMSDS              56265
# MassIngredient              59946
# ClaimantCompany            499401
registry_df_clean.isna().sum()
# registry_df_clean["is_trade_secret"] = registry_df_clean["ClaimantCompany"].notna().astype(int)

DisclosureId                    0
JobStartDate                    0
JobEndDate                      0
APINumber                       0
StateName                       0
CountyName                      0
OperatorName                    0
WellName                        0
Latitude                        0
Longitude                       0
Projection                      0
TVD                         30135
TotalBaseWaterVolume        30157
TotalBaseNonWaterVolume    134538
FFVersion                       0
FederalWell                     0
IndianWell                      0
PurposeId                       0
TradeName                   85580
Supplier                    84129
Purpose                     16499
IngredientsId               56265
CASNumber                   65834
IngredientName              56381
IngredientCommonName       147775
PercentHighAdditive         57761
PercentHFJob                58086
IngredientComment          471882
IngredientMSDS              56265
MassIngredient

In [11]:
registry_columns_to_drop = ['IngredientCommonName',
                            'IngredientComment',
                            'JobStartDate',
                            'JobEndDate',
                            'APINumber',
                            'StateName',
                            'CountyName',
                            'OperatorName',
                            'WellName',
                            'Latitude',
                            'Longitude',
                            'Projection',
                            'TVD',
                            'TotalBaseWaterVolume',
                            'TotalBaseNonWaterVolume',
                            'FFVersion',
                            'FederalWell',
                            'IndianWell']
registry_df_clean = registry_df_clean.drop(columns=registry_columns_to_drop)

## Misc Testing

In [12]:
# is number of wells unique to number disclosure ids?
disclosure_df_clean.isna().sum()

DisclosureId                   0
JobStartDate                  15
JobEndDate                     0
APINumber                      0
StateName                      0
CountyName                     0
OperatorName                   0
WellName                       0
Latitude                      34
Longitude                     34
Projection                     0
TVD                        30139
TotalBaseWaterVolume       30168
TotalBaseNonWaterVolume    50325
FFVersion                      0
FederalWell                    0
IndianWell                     0
dtype: int64

In [13]:
registry_df_clean.isna().sum()

DisclosureId                0
PurposeId                   0
TradeName               85580
Supplier                84129
Purpose                 16499
IngredientsId           56265
CASNumber               65834
IngredientName          56381
PercentHighAdditive     57761
PercentHFJob            58086
IngredientMSDS          56265
MassIngredient          59946
ClaimantCompany        499401
dtype: int64

In [14]:
watersource_df_clean.isna().sum()

WaterSourceId    0
DisclosureId     0
WellName         0
Description      0
Percent          0
dtype: int64